In [ ]:
# Importación de MRJob para implementar trabajos de procesamiento MapReduce.
from mrjob.job import MRJob
import re
import os
import uuid
import hashlib

# Definición de la clase que implementa el trabajo MapReduce.
class MRAnalizadorTXT(MRJob):
    # Configuración de los parámetros de ejecución de Hadoop.
    JOBCONF = {
    # Desactivar la compresión de los datos intermedios generados por Mapper.
        'mapreduce.map.output.compress': False,

        # Desactivar la compresión de los ficheros de salida.
        'mapreduce.output.fileoutputformat.compress': False,

        # Establecer el tiempo máximo de espera de las tareas (600.000 ms).
        'mapreduce.task.timeout': 600000
    }
    # Diccionario que contiene los campos que se deben extraer de los
    # documentos clínicos y sus correspondientes patrones de búsqueda.
    campos = {
    "nombre": r"Nombre:\s*([^\.]+)",
    "apellidos": r"Apellidos:\s*([^\.]+)",
    "CIPA": r"CIPA:\s*([^\.]+)",
    "nass": r"NASS:\s*([^\.]+)",
    "domicilio": r"Domicilio:\s*([^\.]+)",
    "Localidad_Provincia": r"Localidad/ Provincia:\s*([^\.]+)",
    "CP": r"CP:\s*(\d+)",
    "NHC": r"NHC:\s*(\d+)",
    "Fecha_nacimiento": r"Fecha de nacimiento:\s*([^\.]+)",
    "Pais": r"Pa[ií]s:\s*([^\.]+)",
    "Edad": r"Edad:\s*(\d+)",
    "Sexo": r"Sexo:\s*(\w)",
    "Fecha_Ingreso": r"Fecha de Ingreso:\s*([^\.]+)",
    "Especialidad": r"(?:Especialidad|Servicio):\s*([^\.]+)",
    "medico": r"M[eé]dico:\s*([^N]+)",
    "Numero_Colegiado": r"N[ºo]?Col:\s*([0-9\- ]+)",
    "Motivo_ingreso": r"Motivo de ingreso:\s*([^\.]*)",
    "Antecedentes": r"Antecedentes:\s*([^\.]*)",
    "Historia_Actual": r"Historia Actual:\s*(.*?)(?=Diagnostico|Tratamiento|$)",
    "Diagnostico_Principal": r"Diagn[oó]stico Principal:\s*(.*?)(?=Tratamiento:|Remitido por:|$)",
    "Exploracion_fisica_pruebas_complementarias": r"Exploraci[oó]n f[ií]sica y pruebas complementarias:\s*(.*?)(?=Diagn[oó]stico Principal:|Tratamiento:|Remitido por:|>
    "Tratamiento": r"Tratamiento:\s*(.*?)(?=Remitido|$)",
    "Informe_clinico_del_paciente": r"Informe clínico del paciente:\s*(.*?)(?=Diagnostico|Diagnóstico|Tratamiento|Remitido por:|$)"

    }
    contador = 0
 # mapper: lectura y agrupación de las líneas de entrada.
    def mapper(self, _, line):

        import os

        file_path = os.environ.get("map_input_file", "unknown")

        if not line:
           return

        line = line.strip()

        if not line:
           return
# Emitir la ruta del fichero como clave y la línea como valor.
        # Las líneas procedentes del mismo fichero se agruparán en Reduce.
        yield file_path, line


   
    def anonimizar(self, valor):
        # Si el campo no contiene información, devolver una cadena vacía.
        if not valor:
           return ""

        return hashlib.sha256(valor.encode()).hexdigest()[:12]

 # reducer: reconstrucción, extracción y anonimización.
    def reducer(self, key, values):
        # Unir todas las líneas del documento en un único texto.
        # Normalizar los espacios en blanco para facilitar las búsquedas.
        texto = re.sub(r"\s+", " ", " ".join(values))

        datos = {}
        # Incrementar el contador y asignar un identificador al registro.
        MRAnalizadorTXT.contador += 1
        datos["Id"] = MRAnalizadorTXT.contador

        for campo, patron in self.campos.items():
             # Buscar el patrón correspondiente dentro del texto clínico.
            # IGNORECASE permite realizar la búsqueda sin distinguir
            # entre letras mayúsculas y minúsculas.
            m = re.search(patron, texto, re.IGNORECASE)
            datos[campo] = m.group(1).strip() if m else ""

        # Sustituir el valor del campo por un identificador ficticio.
        datos["nombre"] = "Paciente_" + str(datos["Id"])
        datos["apellidos"] = "Apellidos_" + str(self.anonimizar(datos["apellidos"]))
        datos["CIPA"] = self.anonimizar(datos["CIPA"])
        datos["domicilio"] = "Direccion_anonima"

        yield None, datos

# Punto de entrada del programa.
if __name__ == "__main__":
    MRAnalizadorTXT.run()